# 04 — Information-Equivalent Input Representations

## Purpose

This notebook creates paired structured and deterministic natural-language representations of the same sampled network-flow records.

The primary pilot uses 46 model-input fields. These are derived from the provisional 48-field policy by excluding:

- `DST_TO_SRC_SECOND_BYTES`
- `SRC_TO_DST_SECOND_BYTES`

The original 48-field view is retained as a predefined sensitivity condition.

## Experimental controls

Across the structured and natural-language conditions, the following must remain identical:

- sampled records;
- feature names and order;
- underlying values;
- numerical precision;
- missing-value representation;
- class definitions;
- decision instructions;
- output schema; and
- LLM inference settings.

Only the presentation format may change.

Ground-truth labels and attack-category fields must never appear in model inputs.

In [1]:
from pathlib import Path
import hashlib
import json
import platform
import sys

import numpy as np
import pandas as pd


# The notebook normally runs from notebooks/, but this also supports
# execution from the project root.
CURRENT_DIRECTORY = Path.cwd().resolve()

if CURRENT_DIRECTORY.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIRECTORY.parent
else:
    PROJECT_ROOT = CURRENT_DIRECTORY

if not (PROJECT_ROOT / "configs").exists():
    raise FileNotFoundError(
        "Could not locate the project root. "
        "Run this notebook from the project root or the notebooks directory."
    )


FEATURE_POLICY_CSV = (
    PROJECT_ROOT
    / "configs"
    / "feature_policy_provisional.csv"
)

PILOT_FEATURES_CSV = (
    PROJECT_ROOT
    / "data"
    / "interim"
    / "pilot_benign_dos_n200_features.csv"
)

PRIVATE_GROUND_TRUTH_CSV = (
    PROJECT_ROOT
    / "data"
    / "interim"
    / "pilot_benign_dos_n200_ground_truth_private.csv"
)

PILOT_MANIFEST_JSON = (
    PROJECT_ROOT
    / "data"
    / "interim"
    / "pilot_benign_dos_n200_manifest.json"
)


path_check = pd.Series(
    {
        "project_root": str(PROJECT_ROOT),
        "feature_policy_exists": FEATURE_POLICY_CSV.exists(),
        "pilot_features_exists": PILOT_FEATURES_CSV.exists(),
        "private_ground_truth_exists": PRIVATE_GROUND_TRUTH_CSV.exists(),
        "pilot_manifest_exists": PILOT_MANIFEST_JSON.exists(),
        "python_version": platform.python_version(),
        "pandas_version": pd.__version__,
        "numpy_version": np.__version__,
    },
    name="value",
)

path_check

project_root                   /Users/ruiwang/Developer/compsci742-rui-pilot
feature_policy_exists                                                   True
pilot_features_exists                                                   True
private_ground_truth_exists                                             True
pilot_manifest_exists                                                   True
python_version                                                       3.11.14
pandas_version                                                         3.0.5
numpy_version                                                          2.4.6
Name: value, dtype: object

In [2]:
feature_policy = pd.read_csv(FEATURE_POLICY_CSV)
pilot_features = pd.read_csv(PILOT_FEATURES_CSV)
private_ground_truth = pd.read_csv(PRIVATE_GROUND_TRUTH_CSV)

with PILOT_MANIFEST_JSON.open("r", encoding="utf-8") as file:
    pilot_manifest = json.load(file)


# Reconstruct the original provisional 48-field list from the policy file.
sensitivity_48_fields = (
    feature_policy.loc[
        feature_policy["provisional_core_input"].astype(bool)
    ]
    .sort_values("column_position")
    ["column_name"]
    .tolist()
)


# These two fields are excluded from the primary pilot because they contain
# missing or infinite values and were shown to have little effect in the
# team's 51-versus-53 baseline sensitivity analysis.
NONFINITE_RATE_FIELDS = [
    "DST_TO_SRC_SECOND_BYTES",
    "SRC_TO_DST_SECOND_BYTES",
]


primary_46_fields = [
    field
    for field in sensitivity_48_fields
    if field not in NONFINITE_RATE_FIELDS
]


input_summary = pd.Series(
    {
        "policy_rows": len(feature_policy),
        "pilot_feature_rows": len(pilot_features),
        "private_ground_truth_rows": len(private_ground_truth),
        "sensitivity_fields": len(sensitivity_48_fields),
        "primary_fields": len(primary_46_fields),
        "fields_removed_from_primary": len(NONFINITE_RATE_FIELDS),
        "all_sensitivity_fields_in_sample": set(
            sensitivity_48_fields
        ).issubset(pilot_features.columns),
        "all_primary_fields_in_sample": set(
            primary_46_fields
        ).issubset(pilot_features.columns),
    },
    name="value",
)

input_summary

policy_rows                           55
pilot_feature_rows                   200
private_ground_truth_rows            200
sensitivity_fields                    48
primary_fields                        46
fields_removed_from_primary            2
all_sensitivity_fields_in_sample    True
all_primary_fields_in_sample        True
Name: value, dtype: object

In [3]:
removed_fields = sorted(
    set(sensitivity_48_fields) - set(primary_46_fields)
)

unexpected_primary_fields = sorted(
    set(primary_46_fields) - set(sensitivity_48_fields)
)

missing_nonfinite_fields = sorted(
    set(NONFINITE_RATE_FIELDS) - set(sensitivity_48_fields)
)


assert len(sensitivity_48_fields) == 48
assert len(primary_46_fields) == 46
assert removed_fields == sorted(NONFINITE_RATE_FIELDS)
assert unexpected_primary_fields == []
assert missing_nonfinite_fields == []
assert len(primary_46_fields) == len(set(primary_46_fields))
assert len(sensitivity_48_fields) == len(set(sensitivity_48_fields))


feature_set_check = pd.Series(
    {
        "sensitivity_field_count": len(sensitivity_48_fields),
        "primary_field_count": len(primary_46_fields),
        "primary_is_subset_of_sensitivity": set(
            primary_46_fields
        ).issubset(sensitivity_48_fields),
        "removed_fields": ", ".join(removed_fields),
        "unexpected_primary_fields": len(unexpected_primary_fields),
        "duplicate_primary_fields": (
            len(primary_46_fields) - len(set(primary_46_fields))
        ),
        "duplicate_sensitivity_fields": (
            len(sensitivity_48_fields)
            - len(set(sensitivity_48_fields))
        ),
    },
    name="value",
)

feature_set_check

sensitivity_field_count                                                           48
primary_field_count                                                               46
primary_is_subset_of_sensitivity                                                True
removed_fields                      DST_TO_SRC_SECOND_BYTES, SRC_TO_DST_SECOND_BYTES
unexpected_primary_fields                                                          0
duplicate_primary_fields                                                           0
duplicate_sensitivity_fields                                                       0
Name: value, dtype: object

In [4]:
CONFIG_DIRECTORY = PROJECT_ROOT / "configs"

PRIMARY_FEATURE_SET_JSON = (
    CONFIG_DIRECTORY / "feature_set_primary_46.json"
)

SENSITIVITY_FEATURE_SET_JSON = (
    CONFIG_DIRECTORY / "feature_set_sensitivity_48.json"
)


def ordered_field_hash(fields):
    """Return a SHA-256 hash that includes field names and their order."""
    canonical_json = json.dumps(
        fields,
        ensure_ascii=False,
        separators=(",", ":"),
    )
    return hashlib.sha256(
        canonical_json.encode("utf-8")
    ).hexdigest()


primary_feature_config = {
    "schema_version": "1.0",
    "feature_set_id": "primary_46",
    "status": "provisional_pilot",
    "purpose": (
        "Primary feature set for paired structured and deterministic "
        "natural-language LLM input representations."
    ),
    "source_policy": "configs/feature_policy_provisional.csv",
    "source_feature_set": "sensitivity_48",
    "selection_rule": (
        "Use the provisional 48 model-input fields, excluding the two "
        "fields with observed missing or infinite values."
    ),
    "excluded_from_source_feature_set": NONFINITE_RATE_FIELDS,
    "field_count": len(primary_46_fields),
    "ordered_fields_sha256": ordered_field_hash(primary_46_fields),
    "ordered_fields": primary_46_fields,
}


sensitivity_feature_config = {
    "schema_version": "1.0",
    "feature_set_id": "sensitivity_48",
    "status": "predefined_sensitivity",
    "purpose": (
        "Sensitivity feature set retaining the original provisional "
        "48 model-input fields."
    ),
    "source_policy": "configs/feature_policy_provisional.csv",
    "selection_rule": (
        "Use every field marked provisional_core_input in the source "
        "feature-policy file, ordered by source column position."
    ),
    "field_count": len(sensitivity_48_fields),
    "ordered_fields_sha256": ordered_field_hash(
        sensitivity_48_fields
    ),
    "ordered_fields": sensitivity_48_fields,
}


for output_path, payload in [
    (PRIMARY_FEATURE_SET_JSON, primary_feature_config),
    (SENSITIVITY_FEATURE_SET_JSON, sensitivity_feature_config),
]:
    with output_path.open("w", encoding="utf-8") as file:
        json.dump(
            payload,
            file,
            indent=2,
            ensure_ascii=False,
        )
        file.write("\n")


saved_configurations = pd.Series(
    {
        "primary_config": str(
            PRIMARY_FEATURE_SET_JSON.relative_to(PROJECT_ROOT)
        ),
        "primary_exists": PRIMARY_FEATURE_SET_JSON.exists(),
        "primary_field_count": len(primary_46_fields),
        "primary_ordered_fields_sha256": ordered_field_hash(
            primary_46_fields
        ),
        "sensitivity_config": str(
            SENSITIVITY_FEATURE_SET_JSON.relative_to(PROJECT_ROOT)
        ),
        "sensitivity_exists": SENSITIVITY_FEATURE_SET_JSON.exists(),
        "sensitivity_field_count": len(sensitivity_48_fields),
        "sensitivity_ordered_fields_sha256": ordered_field_hash(
            sensitivity_48_fields
        ),
    },
    name="value",
)

saved_configurations

primary_config                                     configs/feature_set_primary_46.json
primary_exists                                                                    True
primary_field_count                                                                 46
primary_ordered_fields_sha256        74b1926e96cffba9209cd3935de7eeb4698e1635d9d9cf...
sensitivity_config                             configs/feature_set_sensitivity_48.json
sensitivity_exists                                                                True
sensitivity_field_count                                                             48
sensitivity_ordered_fields_sha256    d8ed02f46bcad5f06edf3a03a2403d840b3b72a5181947...
Name: value, dtype: object

In [5]:
with PRIMARY_FEATURE_SET_JSON.open(
    "r",
    encoding="utf-8",
) as file:
    reloaded_primary_config = json.load(file)

with SENSITIVITY_FEATURE_SET_JSON.open(
    "r",
    encoding="utf-8",
) as file:
    reloaded_sensitivity_config = json.load(file)


assert reloaded_primary_config["field_count"] == 46
assert reloaded_sensitivity_config["field_count"] == 48

assert (
    reloaded_primary_config["ordered_fields"]
    == primary_46_fields
)

assert (
    reloaded_sensitivity_config["ordered_fields"]
    == sensitivity_48_fields
)

assert (
    reloaded_primary_config["ordered_fields_sha256"]
    == ordered_field_hash(primary_46_fields)
)

assert (
    reloaded_sensitivity_config["ordered_fields_sha256"]
    == ordered_field_hash(sensitivity_48_fields)
)


saved_configuration_check = pd.Series(
    {
        "primary_reload_matches": (
            reloaded_primary_config["ordered_fields"]
            == primary_46_fields
        ),
        "sensitivity_reload_matches": (
            reloaded_sensitivity_config["ordered_fields"]
            == sensitivity_48_fields
        ),
        "primary_hash_matches": (
            reloaded_primary_config["ordered_fields_sha256"]
            == ordered_field_hash(primary_46_fields)
        ),
        "sensitivity_hash_matches": (
            reloaded_sensitivity_config["ordered_fields_sha256"]
            == ordered_field_hash(sensitivity_48_fields)
        ),
        "primary_is_ordered_subset": [
            field
            for field in sensitivity_48_fields
            if field not in NONFINITE_RATE_FIELDS
        ] == primary_46_fields,
    },
    name="value",
)

saved_configuration_check

primary_reload_matches        True
sensitivity_reload_matches    True
primary_hash_matches          True
sensitivity_hash_matches      True
primary_is_ordered_subset     True
Name: value, dtype: bool

## Validate record alignment and label separation

Before generating any LLM inputs, this section verifies that:

1. the sampled feature and private ground-truth tables contain the same records;
2. each `sample_id` is unique;
3. record order is aligned across the two tables;
4. labels and attack categories are absent from model inputs; and
5. the primary 46-field view contains no missing or infinite values.

In [6]:
feature_metadata_columns = [
    column
    for column in pilot_features.columns
    if column not in sensitivity_48_fields
]

ground_truth_columns = private_ground_truth.columns.tolist()


table_structure = pd.Series(
    {
        "pilot_features_shape": str(pilot_features.shape),
        "private_ground_truth_shape": str(
            private_ground_truth.shape
        ),
        "feature_metadata_columns": ", ".join(
            feature_metadata_columns
        ),
        "ground_truth_columns": ", ".join(
            ground_truth_columns
        ),
        "sample_id_in_features": (
            "sample_id" in pilot_features.columns
        ),
        "sample_id_in_ground_truth": (
            "sample_id" in private_ground_truth.columns
        ),
    },
    name="value",
)

table_structure

pilot_features_shape                                                  (200, 50)
private_ground_truth_shape                                             (200, 7)
feature_metadata_columns                            sample_id, model_profile_id
ground_truth_columns          sample_id, model_profile_id, source_row_id, La...
sample_id_in_features                                                      True
sample_id_in_ground_truth                                                  True
Name: value, dtype: object

In [7]:
GROUND_TRUTH_FIELDS = [
    "Label",
    "Attack",
]


assert "sample_id" in pilot_features.columns
assert "sample_id" in private_ground_truth.columns

assert pilot_features["sample_id"].notna().all()
assert private_ground_truth["sample_id"].notna().all()

assert pilot_features["sample_id"].is_unique
assert private_ground_truth["sample_id"].is_unique

assert set(pilot_features["sample_id"]) == set(
    private_ground_truth["sample_id"]
)

assert pilot_features["sample_id"].tolist() == (
    private_ground_truth["sample_id"].tolist()
)

assert not set(GROUND_TRUTH_FIELDS).intersection(
    pilot_features.columns
)

assert not set(GROUND_TRUTH_FIELDS).intersection(
    primary_46_fields
)

assert not set(GROUND_TRUTH_FIELDS).intersection(
    sensitivity_48_fields
)


record_alignment_check = pd.Series(
    {
        "feature_records": len(pilot_features),
        "ground_truth_records": len(private_ground_truth),
        "unique_feature_sample_ids": (
            pilot_features["sample_id"].nunique()
        ),
        "unique_ground_truth_sample_ids": (
            private_ground_truth["sample_id"].nunique()
        ),
        "sample_id_sets_match": (
            set(pilot_features["sample_id"])
            == set(private_ground_truth["sample_id"])
        ),
        "sample_id_order_matches": (
            pilot_features["sample_id"].tolist()
            == private_ground_truth["sample_id"].tolist()
        ),
        "ground_truth_in_feature_table": bool(
            set(GROUND_TRUTH_FIELDS).intersection(
                pilot_features.columns
            )
        ),
        "ground_truth_in_primary_fields": bool(
            set(GROUND_TRUTH_FIELDS).intersection(
                primary_46_fields
            )
        ),
        "ground_truth_in_sensitivity_fields": bool(
            set(GROUND_TRUTH_FIELDS).intersection(
                sensitivity_48_fields
            )
        ),
    },
    name="value",
)

record_alignment_check

feature_records                         200
ground_truth_records                    200
unique_feature_sample_ids               200
unique_ground_truth_sample_ids          200
sample_id_sets_match                   True
sample_id_order_matches                True
ground_truth_in_feature_table         False
ground_truth_in_primary_fields        False
ground_truth_in_sensitivity_fields    False
Name: value, dtype: object

In [8]:
primary_numeric = pilot_features[
    primary_46_fields
].apply(
    pd.to_numeric,
    errors="coerce",
)

sensitivity_numeric = pilot_features[
    sensitivity_48_fields
].apply(
    pd.to_numeric,
    errors="coerce",
)


def count_value_states(frame):
    values = frame.to_numpy(dtype=float)

    return {
        "missing_cells": int(np.isnan(values).sum()),
        "positive_infinite_cells": int(
            np.isposinf(values).sum()
        ),
        "negative_infinite_cells": int(
            np.isneginf(values).sum()
        ),
        "nonfinite_cells": int(
            (~np.isfinite(values)).sum()
        ),
        "records_with_any_nonfinite": int(
            (~np.isfinite(values)).any(axis=1).sum()
        ),
    }


primary_value_states = count_value_states(primary_numeric)
sensitivity_value_states = count_value_states(
    sensitivity_numeric
)


nonfinite_comparison = pd.DataFrame(
    [
        {
            "feature_set": "primary_46",
            **primary_value_states,
        },
        {
            "feature_set": "sensitivity_48",
            **sensitivity_value_states,
        },
    ]
).set_index("feature_set")

nonfinite_comparison

,missing_cells,positive_infinite_cells,negative_infinite_cells,nonfinite_cells,records_with_any_nonfinite
feature_set,,,,,
primary_46,0,0,0,0,0
sensitivity_48,19,29,0,48,24


In [9]:
removed_field_quality = []

for field in NONFINITE_RATE_FIELDS:
    values = pd.to_numeric(
        pilot_features[field],
        errors="coerce",
    ).to_numpy(dtype=float)

    removed_field_quality.append(
        {
            "field": field,
            "missing": int(np.isnan(values).sum()),
            "positive_infinite": int(
                np.isposinf(values).sum()
            ),
            "negative_infinite": int(
                np.isneginf(values).sum()
            ),
            "total_nonfinite": int(
                (~np.isfinite(values)).sum()
            ),
        }
    )


removed_field_quality = pd.DataFrame(
    removed_field_quality
).set_index("field")

removed_field_quality

,missing,positive_infinite,negative_infinite,total_nonfinite
field,,,,
DST_TO_SRC_SECOND_BYTES,0,24,0,24
SRC_TO_DST_SECOND_BYTES,19,5,0,24


## Canonical value-formatting policy

Both input representations must be generated from the same canonical feature-value representation.

The following rules are fixed before LLM inference:

- feature order follows the selected feature-set configuration;
- integer-valued numbers are displayed without a decimal suffix;
- other finite numbers use at most 15 significant digits;
- zero is normalised to `0`;
- missing values use the token `missing`;
- positive infinity uses `positive_infinity`;
- negative infinity uses `negative_infinity`;
- encoded categorical fields remain as raw numeric codes;
- no category meanings, anomaly flags, thresholds or derived interpretations are added;
- provider descriptions and units are not added unless they can be verified and supplied identically to both conditions.

The primary 46-field dataset contains only finite values. The special-value tokens are defined now so that the same formatter can later be used for the 48-field sensitivity analysis.

In [10]:
MISSING_TOKEN = "missing"
POSITIVE_INFINITY_TOKEN = "positive_infinity"
NEGATIVE_INFINITY_TOKEN = "negative_infinity"

SIGNIFICANT_DIGITS = 15


def canonicalize_value(value):
    """
    Convert one stored feature value into a deterministic canonical value.

    Finite values are returned as int or float so that structured JSON
    retains numeric values. Non-finite values are returned as explicit
    string tokens because JSON does not have portable representations
    for NaN or infinity.
    """
    if pd.isna(value):
        return MISSING_TOKEN

    numeric_value = float(value)

    if np.isposinf(numeric_value):
        return POSITIVE_INFINITY_TOKEN

    if np.isneginf(numeric_value):
        return NEGATIVE_INFINITY_TOKEN

    if numeric_value == 0:
        return 0

    if numeric_value.is_integer():
        return int(numeric_value)

    rounded_text = format(
        numeric_value,
        f".{SIGNIFICANT_DIGITS}g",
    )

    return float(rounded_text)


def canonical_value_text(value):
    """
    Return exactly the textual form used when presenting a canonical value.
    """
    canonical_value = canonicalize_value(value)

    if isinstance(canonical_value, str):
        return canonical_value

    return json.dumps(
        canonical_value,
        ensure_ascii=False,
        allow_nan=False,
        separators=(",", ":"),
    )

In [11]:
formatter_test_cases = [
    {
        "input_description": "integer-valued float",
        "input_value": 443.0,
        "expected_canonical": 443,
        "expected_text": "443",
    },
    {
        "input_description": "ordinary decimal",
        "input_value": 12.5,
        "expected_canonical": 12.5,
        "expected_text": "12.5",
    },
    {
        "input_description": "positive zero",
        "input_value": 0.0,
        "expected_canonical": 0,
        "expected_text": "0",
    },
    {
        "input_description": "negative zero",
        "input_value": -0.0,
        "expected_canonical": 0,
        "expected_text": "0",
    },
    {
        "input_description": "missing value",
        "input_value": np.nan,
        "expected_canonical": MISSING_TOKEN,
        "expected_text": MISSING_TOKEN,
    },
    {
        "input_description": "positive infinity",
        "input_value": np.inf,
        "expected_canonical": POSITIVE_INFINITY_TOKEN,
        "expected_text": POSITIVE_INFINITY_TOKEN,
    },
    {
        "input_description": "negative infinity",
        "input_value": -np.inf,
        "expected_canonical": NEGATIVE_INFINITY_TOKEN,
        "expected_text": NEGATIVE_INFINITY_TOKEN,
    },
]


formatter_test_results = []

for test_case in formatter_test_cases:
    actual_canonical = canonicalize_value(
        test_case["input_value"]
    )

    actual_text = canonical_value_text(
        test_case["input_value"]
    )

    canonical_matches = (
        actual_canonical
        == test_case["expected_canonical"]
    )

    text_matches = (
        actual_text
        == test_case["expected_text"]
    )

    assert canonical_matches
    assert text_matches

    formatter_test_results.append(
        {
            "case": test_case["input_description"],
            "canonical_output": actual_canonical,
            "text_output": actual_text,
            "passed": (
                canonical_matches
                and text_matches
            ),
        }
    )


formatter_test_results = pd.DataFrame(
    formatter_test_results
).set_index("case")

formatter_test_results

,canonical_output,text_output,passed
case,,,
integer-valued float,443,443,True
ordinary decimal,12.5,12.5,True
positive zero,0,0,True
negative zero,0,0,True
missing value,missing,missing,True
positive infinity,positive_infinity,positive_infinity,True
negative infinity,negative_infinity,negative_infinity,True


### Interpretation of the formatter tests

All seven boundary tests passed. The formatter therefore produces deterministic outputs for:

- integer-valued numbers;
- decimal values;
- positive and negative zero;
- missing values; and
- positive and negative infinity.

The primary 46-field experiment contains no missing or infinite values. These special-value rules are retained so that the same formatting code can later support the predefined 48-field sensitivity analysis without silently replacing non-finite values with zero or deleting affected records.

In [12]:
def build_canonical_feature_list(row, ordered_fields):
    """
    Build the single source of truth used by both representations.
    """
    return [
        {
            "name": field,
            "value": canonicalize_value(row[field]),
        }
        for field in ordered_fields
    ]


first_sample = pilot_features.iloc[0]

first_canonical_features = build_canonical_feature_list(
    first_sample,
    primary_46_fields,
)


assert len(first_canonical_features) == 46

assert [
    item["name"]
    for item in first_canonical_features
] == primary_46_fields

assert all(
    item["name"] not in GROUND_TRUTH_FIELDS
    for item in first_canonical_features
)


pd.DataFrame(first_canonical_features).head(10)

,name,value
0,L4_SRC_PORT,47350
1,L4_DST_PORT,1581
2,PROTOCOL,6
3,L7_PROTO,0
4,IN_BYTES,492
5,IN_PKTS,10
6,OUT_BYTES,504
7,OUT_PKTS,10
8,TCP_FLAGS,19
9,CLIENT_TCP_FLAGS,19


## Define the paired representation templates

Each sampled record is first converted into one canonical ordered feature list. Both presentation conditions are rendered from that same list.

### Structured condition

The structured condition uses valid JSON containing an ordered list of feature names and values.

### Deterministic natural-language condition

The text condition uses one fixed sentence template per feature:

`Feature <name> has value <value>.`

No feature descriptions, thresholds, anomaly hints or interpretations are added. The `sample_id` remains outside the model input and is used only to link experiment outputs back to the sampled record.

### Implementation logic

The code below separates the process into four stages:

1. `build_canonical_feature_list` creates one ordered feature–value record that acts as the internal source of truth.
2. `render_structured_input` converts that canonical record into JSON.
3. `render_text_input` converts the same canonical record into deterministic sentences.
4. `parse_structured_input` and `parse_text_input` convert the two rendered inputs back into ordered feature–value pairs for automated comparison.

This design prevents the two conditions from independently reading or transforming the source data. Any difference between them should therefore come from presentation format rather than different feature selection, value formatting or field order.

In [13]:
TEXT_HEADER = "Network-flow record."


def canonical_output_text(canonical_value):
    """
    Convert an already-canonical value into its exact display token.
    """
    if isinstance(canonical_value, str):
        return canonical_value

    return json.dumps(
        canonical_value,
        ensure_ascii=False,
        allow_nan=False,
        separators=(",", ":"),
    )


def build_canonical_feature_list(row, ordered_fields):
    """
    Create the single ordered source of truth for one record.
    """
    return [
        {
            "name": field,
            "value": canonicalize_value(row[field]),
        }
        for field in ordered_fields
    ]


def render_structured_input(canonical_features):
    """
    Render one canonical record as valid, deterministic JSON.
    """
    payload = {
        "record_type": "network_flow",
        "features": canonical_features,
    }

    return json.dumps(
        payload,
        ensure_ascii=False,
        allow_nan=False,
        separators=(",", ":"),
    )


def render_text_input(canonical_features):
    """
    Render one canonical record as deterministic natural-language text.
    """
    lines = [TEXT_HEADER]

    lines.extend(
        (
            f"Feature {item['name']} has value "
            f"{canonical_output_text(item['value'])}."
        )
        for item in canonical_features
    )

    return "\n".join(lines)

In [14]:
first_sample = pilot_features.iloc[0]

first_canonical_features = build_canonical_feature_list(
    first_sample,
    primary_46_fields,
)

first_structured_input = render_structured_input(
    first_canonical_features
)

first_text_input = render_text_input(
    first_canonical_features
)


assert len(first_canonical_features) == 46

assert [
    item["name"]
    for item in first_canonical_features
] == primary_46_fields


print("Sample ID:")
print(first_sample["sample_id"])

print("\nCanonical feature count:")
print(len(first_canonical_features))

print("\nStructured preview:")
print(first_structured_input[:1000])

print("\nNatural-language preview:")
print(first_text_input[:1000])

Sample ID:
pilot_001

Canonical feature count:
46

Structured preview:
{"record_type":"network_flow","features":[{"name":"L4_SRC_PORT","value":47350},{"name":"L4_DST_PORT","value":1581},{"name":"PROTOCOL","value":6},{"name":"L7_PROTO","value":0},{"name":"IN_BYTES","value":492},{"name":"IN_PKTS","value":10},{"name":"OUT_BYTES","value":504},{"name":"OUT_PKTS","value":10},{"name":"TCP_FLAGS","value":19},{"name":"CLIENT_TCP_FLAGS","value":19},{"name":"SERVER_TCP_FLAGS","value":19},{"name":"FLOW_DURATION_MILLISECONDS","value":857},{"name":"DURATION_IN","value":857},{"name":"DURATION_OUT","value":788},{"name":"MIN_TTL","value":254},{"name":"MAX_TTL","value":255},{"name":"LONGEST_FLOW_PKT","value":84},{"name":"SHORTEST_FLOW_PKT","value":40},{"name":"MIN_IP_PKT_LEN","value":40},{"name":"MAX_IP_PKT_LEN","value":84},{"name":"RETRANSMITTED_IN_BYTES","value":118},{"name":"RETRANSMITTED_IN_PKTS","value":2},{"name":"RETRANSMITTED_OUT_BYTES","value":84},{"name":"RETRANSMITTED_OUT_PKTS","value":1},{"n

In [15]:
def parse_structured_input(structured_input):
    """
    Parse a structured representation back into ordered name-value tokens.
    """
    payload = json.loads(structured_input)

    assert payload["record_type"] == "network_flow"

    return [
        (
            item["name"],
            canonical_output_text(item["value"]),
        )
        for item in payload["features"]
    ]


def parse_text_input(text_input):
    """
    Parse deterministic text back into ordered name-value tokens.
    """
    lines = text_input.splitlines()

    assert lines[0] == TEXT_HEADER

    parsed_features = []

    for line in lines[1:]:
        assert line.startswith("Feature ")
        assert line.endswith(".")
        assert " has value " in line

        content = line[
            len("Feature "):-1
        ]

        feature_name, value_text = content.split(
            " has value ",
            maxsplit=1,
        )

        parsed_features.append(
            (feature_name, value_text)
        )

    return parsed_features

In [16]:
first_structured_pairs = parse_structured_input(
    first_structured_input
)

first_text_pairs = parse_text_input(
    first_text_input
)

expected_pairs = [
    (
        item["name"],
        canonical_output_text(item["value"]),
    )
    for item in first_canonical_features
]


assert first_structured_pairs == expected_pairs
assert first_text_pairs == expected_pairs
assert first_structured_pairs == first_text_pairs


first_equivalence_check = pd.Series(
    {
        "sample_id": first_sample["sample_id"],
        "expected_feature_count": len(expected_pairs),
        "structured_feature_count": len(
            first_structured_pairs
        ),
        "text_feature_count": len(first_text_pairs),
        "structured_matches_canonical": (
            first_structured_pairs == expected_pairs
        ),
        "text_matches_canonical": (
            first_text_pairs == expected_pairs
        ),
        "structured_matches_text": (
            first_structured_pairs == first_text_pairs
        ),
        "label_present": (
            "Label" in first_structured_input
            or "Label" in first_text_input
        ),
        "attack_present": (
            "Attack" in first_structured_input
            or "Attack" in first_text_input
        ),
    },
    name="value",
)

first_equivalence_check

sample_id                       pilot_001
expected_feature_count                 46
structured_feature_count               46
text_feature_count                     46
structured_matches_canonical         True
text_matches_canonical               True
structured_matches_text              True
label_present                       False
attack_present                      False
Name: value, dtype: object

### Interpretation of the first-record equivalence check

The first sampled record, `pilot_001`, contains 46 features in the canonical, structured and natural-language representations.

After both rendered inputs were parsed back into ordered feature–value pairs:

- the structured representation exactly matched the canonical record;
- the natural-language representation exactly matched the canonical record;
- the two rendered representations matched each other; and
- neither `Label` nor `Attack` appeared in either model input.

This check confirms information equivalence for the first record only. The same validation must next be applied to all 200 sampled records.

## Generate and validate all 200 paired inputs

The first-record check demonstrated that the rendering and parsing functions work for one example. This section applies the same process to every sampled record.

For each record, the notebook:

1. builds one canonical ordered list of 46 feature–value pairs;
2. renders that list as structured JSON;
3. renders the same list as deterministic natural-language text;
4. parses both rendered inputs back into feature–value pairs;
5. compares both parsed results with the canonical record;
6. confirms that both conditions match each other;
7. confirms that excluded fields and `sample_id` are absent from the model input; and
8. calculates a SHA-256 hash for the canonical payload.

The hash identifies the underlying information supplied in the two conditions. Matching hashes do not replace the field-by-field equivalence checks; they provide an additional reproducibility identifier.

No files are written in this section. Outputs are saved only after all 200 records pass validation.

In [17]:
def canonical_payload_hash(canonical_features):
    """
    Hash the ordered canonical feature list for one sampled record.
    """
    canonical_json = json.dumps(
        canonical_features,
        ensure_ascii=False,
        allow_nan=False,
        separators=(",", ":"),
    )

    return hashlib.sha256(
        canonical_json.encode("utf-8")
    ).hexdigest()


# Fields excluded by the original policy, plus the two fields removed
# from the primary 46-field view, must never appear in primary inputs.
policy_excluded_fields = set(
    feature_policy.loc[
        ~feature_policy["provisional_core_input"].astype(bool),
        "column_name",
    ]
)

forbidden_primary_fields = (
    policy_excluded_fields
    | set(NONFINITE_RATE_FIELDS)
)


structured_records = []
text_records = []
validation_records = []


for _, row in pilot_features.iterrows():
    sample_id = row["sample_id"]

    # Stage 1: create the single internal source of truth.
    canonical_features = build_canonical_feature_list(
        row,
        primary_46_fields,
    )

    expected_pairs = [
        (
            item["name"],
            canonical_output_text(item["value"]),
        )
        for item in canonical_features
    ]

    # Stage 2: render the same source record in two formats.
    structured_input = render_structured_input(
        canonical_features
    )

    text_input = render_text_input(
        canonical_features
    )

    # Stage 3: parse both rendered inputs back into comparable pairs.
    structured_pairs = parse_structured_input(
        structured_input
    )

    text_pairs = parse_text_input(
        text_input
    )

    structured_field_names = {
        name
        for name, _ in structured_pairs
    }

    text_field_names = {
        name
        for name, _ in text_pairs
    }

    forbidden_in_structured = sorted(
        structured_field_names
        & forbidden_primary_fields
    )

    forbidden_in_text = sorted(
        text_field_names
        & forbidden_primary_fields
    )

    payload_sha256 = canonical_payload_hash(
        canonical_features
    )

    # These outer metadata fields are saved for experiment management.
    # Only the value of model_input will later be sent to the LLM.
    structured_records.append(
        {
            "sample_id": sample_id,
            "feature_set_id": "primary_46",
            "canonical_payload_sha256": payload_sha256,
            "model_input": structured_input,
        }
    )

    text_records.append(
        {
            "sample_id": sample_id,
            "feature_set_id": "primary_46",
            "canonical_payload_sha256": payload_sha256,
            "model_input": text_input,
        }
    )

    validation_records.append(
        {
            "sample_id": sample_id,
            "canonical_feature_count": len(
                expected_pairs
            ),
            "structured_feature_count": len(
                structured_pairs
            ),
            "text_feature_count": len(
                text_pairs
            ),
            "structured_matches_canonical": (
                structured_pairs == expected_pairs
            ),
            "text_matches_canonical": (
                text_pairs == expected_pairs
            ),
            "structured_matches_text": (
                structured_pairs == text_pairs
            ),
            "structured_forbidden_field_count": len(
                forbidden_in_structured
            ),
            "text_forbidden_field_count": len(
                forbidden_in_text
            ),
            "sample_id_inside_structured_input": (
                str(sample_id) in structured_input
            ),
            "sample_id_inside_text_input": (
                str(sample_id) in text_input
            ),
        }
    )


structured_records = pd.DataFrame(
    structured_records
)

text_records = pd.DataFrame(
    text_records
)

representation_validation = pd.DataFrame(
    validation_records
)

In [18]:
BOOLEAN_VALIDATION_COLUMNS = [
    "structured_matches_canonical",
    "text_matches_canonical",
    "structured_matches_text",
]

LEAKAGE_VALIDATION_COLUMNS = [
    "sample_id_inside_structured_input",
    "sample_id_inside_text_input",
]


assert len(structured_records) == 200
assert len(text_records) == 200
assert len(representation_validation) == 200

assert structured_records["sample_id"].is_unique
assert text_records["sample_id"].is_unique

assert structured_records["sample_id"].tolist() == (
    text_records["sample_id"].tolist()
)

assert (
    structured_records["canonical_payload_sha256"].tolist()
    == text_records["canonical_payload_sha256"].tolist()
)

assert (
    representation_validation[
        "canonical_feature_count"
    ] == 46
).all()

assert (
    representation_validation[
        "structured_feature_count"
    ] == 46
).all()

assert (
    representation_validation[
        "text_feature_count"
    ] == 46
).all()

assert representation_validation[
    BOOLEAN_VALIDATION_COLUMNS
].all().all()

assert (
    representation_validation[
        "structured_forbidden_field_count"
    ] == 0
).all()

assert (
    representation_validation[
        "text_forbidden_field_count"
    ] == 0
).all()

assert not representation_validation[
    LEAKAGE_VALIDATION_COLUMNS
].any().any()


all_record_validation_summary = pd.Series(
    {
        "records_checked": len(
            representation_validation
        ),
        "canonical_count_correct": int(
            (
                representation_validation[
                    "canonical_feature_count"
                ] == 46
            ).sum()
        ),
        "structured_count_correct": int(
            (
                representation_validation[
                    "structured_feature_count"
                ] == 46
            ).sum()
        ),
        "text_count_correct": int(
            (
                representation_validation[
                    "text_feature_count"
                ] == 46
            ).sum()
        ),
        "structured_matches_canonical": int(
            representation_validation[
                "structured_matches_canonical"
            ].sum()
        ),
        "text_matches_canonical": int(
            representation_validation[
                "text_matches_canonical"
            ].sum()
        ),
        "structured_matches_text": int(
            representation_validation[
                "structured_matches_text"
            ].sum()
        ),
        "records_with_forbidden_fields": int(
            (
                (
                    representation_validation[
                        "structured_forbidden_field_count"
                    ]
                    + representation_validation[
                        "text_forbidden_field_count"
                    ]
                ) > 0
            ).sum()
        ),
        "records_with_sample_id_in_input": int(
            representation_validation[
                LEAKAGE_VALIDATION_COLUMNS
            ].any(axis=1).sum()
        ),
        "matching_payload_hashes": int(
            (
                structured_records[
                    "canonical_payload_sha256"
                ]
                == text_records[
                    "canonical_payload_sha256"
                ]
            ).sum()
        ),
    },
    name="value",
)

all_record_validation_summary

records_checked                    200
canonical_count_correct            200
structured_count_correct           200
text_count_correct                 200
structured_matches_canonical       200
text_matches_canonical             200
structured_matches_text            200
records_with_forbidden_fields        0
records_with_sample_id_in_input      0
matching_payload_hashes            200
Name: value, dtype: int64

### Interpretation of the full-sample equivalence check

All 200 sampled records passed the automated representation checks.

For every record:

- the canonical, structured and natural-language versions contained exactly 46 ordered feature–value pairs;
- the structured representation parsed back to the canonical record;
- the natural-language representation parsed back to the canonical record;
- the two representations contained identical underlying information;
- no excluded field appeared in either model input;
- `sample_id` remained outside the model input; and
- the two conditions shared the same canonical payload hash.

The paired inputs are therefore ready to be saved as reproducible intermediate artifacts. This validation establishes information equivalence under the deterministic templates used in this notebook; it does not imply that the two representations will use the same number of tokens or produce the same LLM behaviour.

## Save the validated primary representations

Only after all 200 records pass the equivalence and leakage checks are the paired inputs written to disk.

Three intermediate artifacts are saved:

1. `structured.jsonl` contains the structured JSON condition;
2. `deterministic_text.jsonl` contains the natural-language condition; and
3. `equivalence_validation.csv` records the per-sample validation results.

A fourth file, `manifest.json`, records the feature set, formatting rules, template versions, record counts and SHA-256 hashes of the generated files.

The representation files contain `sample_id` as outer experiment metadata, but `sample_id` is not included inside `model_input`. No ground-truth label or attack category is written to either representation file.

In [19]:
PRIMARY_OUTPUT_DIRECTORY = (
    PROJECT_ROOT
    / "data"
    / "interim"
    / "representations"
    / "primary_46"
)

PRIMARY_OUTPUT_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True,
)


STRUCTURED_JSONL = (
    PRIMARY_OUTPUT_DIRECTORY / "structured.jsonl"
)

TEXT_JSONL = (
    PRIMARY_OUTPUT_DIRECTORY / "deterministic_text.jsonl"
)

VALIDATION_CSV = (
    PRIMARY_OUTPUT_DIRECTORY / "equivalence_validation.csv"
)

REPRESENTATION_MANIFEST_JSON = (
    PRIMARY_OUTPUT_DIRECTORY / "manifest.json"
)


def write_jsonl(records, output_path):
    """
    Write one dictionary per line as valid JSON.
    """
    with output_path.open(
        "w",
        encoding="utf-8",
    ) as file:
        for record in records:
            line = json.dumps(
                record,
                ensure_ascii=False,
                allow_nan=False,
                separators=(",", ":"),
            )
            file.write(line + "\n")


def file_sha256(file_path):
    """
    Calculate a SHA-256 hash for a completed file.
    """
    sha256 = hashlib.sha256()

    with file_path.open("rb") as file:
        for chunk in iter(
            lambda: file.read(1024 * 1024),
            b"",
        ):
            sha256.update(chunk)

    return sha256.hexdigest()


# Save the two paired model-input conditions.
write_jsonl(
    structured_records.to_dict(orient="records"),
    STRUCTURED_JSONL,
)

write_jsonl(
    text_records.to_dict(orient="records"),
    TEXT_JSONL,
)


# Save the detailed validation table separately.
representation_validation.to_csv(
    VALIDATION_CSV,
    index=False,
    lineterminator="\n",
)


representation_manifest = {
    "schema_version": "1.0",
    "artifact_set_id": "primary_46_paired_representations",
    "generated_by": "notebooks/04_input_representation.ipynb",
    "feature_set_id": "primary_46",
    "feature_set_config": (
        "configs/feature_set_primary_46.json"
    ),
    "feature_set_config_sha256": file_sha256(
        PRIMARY_FEATURE_SET_JSON
    ),
    "source_sample": (
        "data/interim/"
        "pilot_benign_dos_n200_features.csv"
    ),
    "source_sample_sha256": file_sha256(
        PILOT_FEATURES_CSV
    ),
    "record_count": len(structured_records),
    "feature_count_per_record": 46,
    "formatter": {
        "significant_digits": SIGNIFICANT_DIGITS,
        "integer_valued_numbers_without_decimal": True,
        "zero_token": "0",
        "missing_token": MISSING_TOKEN,
        "positive_infinity_token": (
            POSITIVE_INFINITY_TOKEN
        ),
        "negative_infinity_token": (
            NEGATIVE_INFINITY_TOKEN
        ),
    },
    "templates": {
        "structured": {
            "format": "JSON",
            "record_type": "network_flow",
            "ordered_feature_objects": True,
        },
        "deterministic_text": {
            "header": TEXT_HEADER,
            "feature_sentence": (
                "Feature <name> has value <value>."
            ),
        },
    },
    "validation": {
        "records_checked": len(
            representation_validation
        ),
        "structured_matches_canonical": int(
            representation_validation[
                "structured_matches_canonical"
            ].sum()
        ),
        "text_matches_canonical": int(
            representation_validation[
                "text_matches_canonical"
            ].sum()
        ),
        "structured_matches_text": int(
            representation_validation[
                "structured_matches_text"
            ].sum()
        ),
        "records_with_forbidden_fields": 0,
        "records_with_sample_id_in_model_input": 0,
    },
    "files": {
        "structured.jsonl": {
            "sha256": file_sha256(
                STRUCTURED_JSONL
            ),
        },
        "deterministic_text.jsonl": {
            "sha256": file_sha256(
                TEXT_JSONL
            ),
        },
        "equivalence_validation.csv": {
            "sha256": file_sha256(
                VALIDATION_CSV
            ),
        },
    },
}


with REPRESENTATION_MANIFEST_JSON.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        representation_manifest,
        file,
        indent=2,
        ensure_ascii=False,
    )
    file.write("\n")


saved_artifacts = pd.DataFrame(
    [
        {
            "artifact": path.name,
            "relative_path": str(
                path.relative_to(PROJECT_ROOT)
            ),
            "size_bytes": path.stat().st_size,
            "sha256": file_sha256(path),
        }
        for path in [
            STRUCTURED_JSONL,
            TEXT_JSONL,
            VALIDATION_CSV,
            REPRESENTATION_MANIFEST_JSON,
        ]
    ]
).set_index("artifact")

saved_artifacts

,relative_path,size_bytes,sha256
artifact,,,
structured.jsonl,data/interim/representations/primary_46/struct...,462428,a74fdbb25e8d24fb9afd06d13982f19689a04685436a65...
deterministic_text.jsonl,data/interim/representations/primary_46/determ...,410628,e47549bc5c5ae08d170e46c43b223afdaae6d3aaef7106...
equivalence_validation.csv,data/interim/representations/primary_46/equiva...,10276,a3c036f068e2dcc2a502b25ac34c390daa6c20f0c0bcc3...
manifest.json,data/interim/representations/primary_46/manife...,1745,c653d611be838dc7dfb98b04ab22a390e61a38eed1c5dd...


## Reload and verify the saved artifacts

The preceding checks validated the representations in memory. This section independently reloads the saved files from disk.

This distinction matters because a correct in-memory object can still be damaged during serialisation or file writing. For example, a JSONL file could lose a line, a multiline text value could be incorrectly escaped, or a manifest could contain a stale hash.

The reloaded artifacts must satisfy the following conditions:

- both JSONL files contain exactly 200 valid records;
- their `sample_id` values are unique and appear in the same order;
- paired records contain the same canonical payload hash;
- every saved model input still parses into exactly 46 feature–value pairs;
- each saved structured input matches its paired text input;
- the saved validation table contains 200 records; and
- the actual file hashes match those recorded in the manifest.

In [20]:
def read_jsonl(input_path):
    """
    Read a JSONL file and return one dictionary per non-empty line.
    """
    records = []

    with input_path.open(
        "r",
        encoding="utf-8",
    ) as file:
        for line_number, line in enumerate(
            file,
            start=1,
        ):
            stripped_line = line.strip()

            if not stripped_line:
                continue

            try:
                records.append(
                    json.loads(stripped_line)
                )
            except json.JSONDecodeError as error:
                raise ValueError(
                    f"Invalid JSON on line {line_number} "
                    f"of {input_path.name}"
                ) from error

    return records


reloaded_structured_records = pd.DataFrame(
    read_jsonl(STRUCTURED_JSONL)
)

reloaded_text_records = pd.DataFrame(
    read_jsonl(TEXT_JSONL)
)

reloaded_validation = pd.read_csv(
    VALIDATION_CSV
)

with REPRESENTATION_MANIFEST_JSON.open(
    "r",
    encoding="utf-8",
) as file:
    reloaded_representation_manifest = json.load(file)


reloaded_shapes = pd.Series(
    {
        "structured_shape": str(
            reloaded_structured_records.shape
        ),
        "text_shape": str(
            reloaded_text_records.shape
        ),
        "validation_shape": str(
            reloaded_validation.shape
        ),
        "manifest_record_count": (
            reloaded_representation_manifest[
                "record_count"
            ]
        ),
    },
    name="value",
)

reloaded_shapes

structured_shape          (200, 4)
text_shape                (200, 4)
validation_shape         (200, 11)
manifest_record_count          200
Name: value, dtype: object

In [21]:
assert len(reloaded_structured_records) == 200
assert len(reloaded_text_records) == 200
assert len(reloaded_validation) == 200

assert reloaded_structured_records[
    "sample_id"
].is_unique

assert reloaded_text_records[
    "sample_id"
].is_unique

assert (
    reloaded_structured_records[
        "sample_id"
    ].tolist()
    == reloaded_text_records[
        "sample_id"
    ].tolist()
)

assert (
    reloaded_structured_records[
        "canonical_payload_sha256"
    ].tolist()
    == reloaded_text_records[
        "canonical_payload_sha256"
    ].tolist()
)


reloaded_pair_checks = []

for structured_record, text_record in zip(
    reloaded_structured_records.to_dict(
        orient="records"
    ),
    reloaded_text_records.to_dict(
        orient="records"
    ),
    strict=True,
):
    structured_pairs = parse_structured_input(
        structured_record["model_input"]
    )

    text_pairs = parse_text_input(
        text_record["model_input"]
    )

    reloaded_pair_checks.append(
        {
            "sample_id": structured_record[
                "sample_id"
            ],
            "sample_ids_match": (
                structured_record["sample_id"]
                == text_record["sample_id"]
            ),
            "payload_hashes_match": (
                structured_record[
                    "canonical_payload_sha256"
                ]
                == text_record[
                    "canonical_payload_sha256"
                ]
            ),
            "structured_feature_count": len(
                structured_pairs
            ),
            "text_feature_count": len(
                text_pairs
            ),
            "representations_match": (
                structured_pairs == text_pairs
            ),
        }
    )


reloaded_pair_checks = pd.DataFrame(
    reloaded_pair_checks
)


assert reloaded_pair_checks[
    "sample_ids_match"
].all()

assert reloaded_pair_checks[
    "payload_hashes_match"
].all()

assert (
    reloaded_pair_checks[
        "structured_feature_count"
    ] == 46
).all()

assert (
    reloaded_pair_checks[
        "text_feature_count"
    ] == 46
).all()

assert reloaded_pair_checks[
    "representations_match"
].all()

In [22]:
manifest_files = (
    reloaded_representation_manifest["files"]
)

actual_file_hashes = {
    "structured.jsonl": file_sha256(
        STRUCTURED_JSONL
    ),
    "deterministic_text.jsonl": file_sha256(
        TEXT_JSONL
    ),
    "equivalence_validation.csv": file_sha256(
        VALIDATION_CSV
    ),
}


hash_check_rows = []

for filename, actual_hash in actual_file_hashes.items():
    expected_hash = manifest_files[
        filename
    ]["sha256"]

    hash_check_rows.append(
        {
            "filename": filename,
            "expected_sha256": expected_hash,
            "actual_sha256": actual_hash,
            "matches": expected_hash == actual_hash,
        }
    )


saved_hash_check = pd.DataFrame(
    hash_check_rows
).set_index("filename")

assert saved_hash_check["matches"].all()

saved_hash_check

,expected_sha256,actual_sha256,matches
filename,,,
structured.jsonl,a74fdbb25e8d24fb9afd06d13982f19689a04685436a65...,a74fdbb25e8d24fb9afd06d13982f19689a04685436a65...,True
deterministic_text.jsonl,e47549bc5c5ae08d170e46c43b223afdaae6d3aaef7106...,e47549bc5c5ae08d170e46c43b223afdaae6d3aaef7106...,True
equivalence_validation.csv,a3c036f068e2dcc2a502b25ac34c390daa6c20f0c0bcc3...,a3c036f068e2dcc2a502b25ac34c390daa6c20f0c0bcc3...,True


In [23]:
disk_verification_summary = pd.Series(
    {
        "structured_records_reloaded": len(
            reloaded_structured_records
        ),
        "text_records_reloaded": len(
            reloaded_text_records
        ),
        "validation_records_reloaded": len(
            reloaded_validation
        ),
        "sample_id_pairs_matching": int(
            reloaded_pair_checks[
                "sample_ids_match"
            ].sum()
        ),
        "payload_hash_pairs_matching": int(
            reloaded_pair_checks[
                "payload_hashes_match"
            ].sum()
        ),
        "structured_records_with_46_fields": int(
            (
                reloaded_pair_checks[
                    "structured_feature_count"
                ] == 46
            ).sum()
        ),
        "text_records_with_46_fields": int(
            (
                reloaded_pair_checks[
                    "text_feature_count"
                ] == 46
            ).sum()
        ),
        "representation_pairs_matching": int(
            reloaded_pair_checks[
                "representations_match"
            ].sum()
        ),
        "artifact_file_hashes_matching": int(
            saved_hash_check["matches"].sum()
        ),
        "artifact_files_checked": len(
            saved_hash_check
        ),
    },
    name="value",
)

disk_verification_summary

structured_records_reloaded          200
text_records_reloaded                200
validation_records_reloaded          200
sample_id_pairs_matching             200
payload_hash_pairs_matching          200
structured_records_with_46_fields    200
text_records_with_46_fields          200
representation_pairs_matching        200
artifact_file_hashes_matching          3
artifact_files_checked                 3
Name: value, dtype: int64

### Interpretation of the disk verification

The saved primary representation artifacts were successfully reloaded and independently revalidated.

Both JSONL files contained 200 uniquely identified records in the same order. Every paired record retained the same canonical payload hash, contained 46 feature–value pairs, and produced identical parsed information across the structured and deterministic natural-language conditions.

The detailed validation table also contained 200 records, and the SHA-256 hashes of all three data artifacts matched the values stored in the manifest.

The serialisation and file-writing process therefore did not alter the validated model inputs.

## Conclusion

This notebook produced a validated primary input-representation dataset for the 200-record Benign-versus-DoS pilot.

The primary condition uses 46 fields. The same records, field names, field order, canonical values and numerical precision were rendered as:

1. structured JSON; and
2. deterministic natural-language text.

Automated forward-rendering, reverse-parsing, leakage and disk-integrity checks confirmed that all 200 representation pairs contain the same underlying information and exclude ground-truth fields.

The original provisional 48-field configuration has been retained as a predefined sensitivity condition, but its LLM inputs are not generated in this notebook run. The next notebook will define the common LLM instructions, output schema and inference settings before any model requests are sent.